<a href="https://colab.research.google.com/github/anarghasunil2014-eng/My-projects-in-RNN/blob/main/Name_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np

# ----------------------------------
# Dataset
# ----------------------------------

names = ["anu","arun","akhil","anjali"]

chars = sorted(list(set("".join(names))))
chars.append("<END>")

char_to_idx = {ch:i for i,ch in enumerate(chars)}
idx_to_char = {i:ch for ch,i in char_to_idx.items()}

vocab_size = len(chars)

# ----------------------------------
# Hyperparameters
# ----------------------------------

hidden_size = 32
lr = 0.01

# ----------------------------------
# Weights
# ----------------------------------

Wxh = np.random.randn(hidden_size,vocab_size)*0.01
Whh = np.random.randn(hidden_size,hidden_size)*0.01

Why = np.random.randn(vocab_size,hidden_size)*0.01

bh = np.zeros((hidden_size,1))
by = np.zeros((vocab_size,1))

In [4]:
def softmax(x):

    exp_x = np.exp(x - np.max(x))

    return exp_x / np.sum(exp_x)

In [5]:
def forward(inputs,targets):

    xs={}
    hs={}
    ys={}
    ps={}

    hs[-1]=np.zeros((hidden_size,1))

    loss=0

    for t in range(len(inputs)):

        xs[t]=np.zeros((vocab_size,1))
        xs[t][inputs[t]]=1

        hs[t]=np.tanh(
            np.dot(Wxh,xs[t])
            +
            np.dot(Whh,hs[t-1])
            +
            bh
        )

        ys[t]=np.dot(Why,hs[t])+by

        ps[t]=softmax(ys[t])

        loss += -np.log(ps[t][targets[t],0])

    return loss,xs,hs,ps

In [6]:
def backward(inputs,targets,xs,hs,ps):

    dWxh=np.zeros_like(Wxh)
    dWhh=np.zeros_like(Whh)
    dWhy=np.zeros_like(Why)

    dbh=np.zeros_like(bh)
    dby=np.zeros_like(by)

    dh_next=np.zeros((hidden_size,1))

    for t in reversed(range(len(inputs))):

        dy=np.copy(ps[t])

        dy[targets[t]] -= 1

        dWhy += np.dot(dy,hs[t].T)

        dby += dy

        dh=np.dot(Why.T,dy)+dh_next

        dh_raw=(1-hs[t]**2)*dh

        dbh += dh_raw

        dWxh += np.dot(dh_raw,xs[t].T)

        dWhh += np.dot(dh_raw,hs[t-1].T)

        dh_next=np.dot(Whh.T,dh_raw)

    return dWxh,dWhh,dWhy,dbh,dby

In [7]:
# --- Minimal Forward/Backward Pass (for demonstration of gradient definition) ---
# In a full training loop, 'inputs' and 'targets' would come from your dataset
# and these steps would be inside your main training loop.

# Example: Process a single name 'anu'
example_name = names[0] # 'anu'
inputs = [char_to_idx[ch] for ch in example_name[:-1]]
targets = [char_to_idx[ch] for ch in example_name[1:]] + [char_to_idx['<END>']]

# Perform forward pass to get intermediate states
loss, xs, hs, ps = forward(inputs, targets)

# Perform backward pass to compute gradients
dWxh, dWhh, dWhy, dbh, dby = backward(inputs, targets, xs, hs, ps)

# --- Weight Update Step ---
Wxh -= lr*dWxh
Whh -= lr*dWhh

Why -= lr*dWhy

bh -= lr*dbh
by -= lr*dby

In [11]:
for epoch in range(1000):

    total_loss = 0

    for word in names:

        # Convert word to list of characters, then append '<END>' token
        token_list = list(word) + ["<END>"]

        inputs = [char_to_idx[token]
                  for token in token_list[:-1]]

        targets = [char_to_idx[token]
                   for token in token_list[1:]]

        loss,xs,hs,ps = forward(
            inputs,
            targets
        )

        grads = backward(
            inputs,
            targets,
            xs,
            hs,
            ps
        )

        dWxh,dWhh,dWhy,dbh,dby = grads

        Wxh -= lr*dWxh
        Whh -= lr*dWhh
        Why -= lr*dWhy

        bh -= lr*dbh
        by -= lr*dby

        total_loss += loss

    print(epoch,total_loss)

0 38.48356646269456
1 38.45478069291172
2 38.42491706238337
3 38.393930257316654
4 38.361773994433676
5 38.32840104571154
6 38.293763269121385
7 38.25781164558209
8 38.22049632229566
9 38.18176666257125
10 38.14157130217111
11 38.09985821212355
12 38.05657476784362
13 38.01166782428096
14 37.96508379667497
15 37.916768746338235
16 37.86666847071171
17 37.8147285967365
18 37.76089467636998
19 37.705112282837845
20 37.64732710596081
21 37.58748504462781
22 37.5255322942098
23 37.46141542642573
24 37.395081458890175
25 37.32647791130001
26 37.25555284496467
27 37.182254882163434
28 37.1065332016387
29 37.02833750642326
30 36.94761796017133
31 36.86432508824045
32 36.778409639978705
33 36.68982240903616
34 36.59851400906982
35 36.504434602979664
36 36.407533584829416
37 36.30775921490175
38 36.20505820994387
39 36.0993752926013
40 35.99065270633727
41 35.87882970480537
42 35.76384202768568
43 35.64562137839707
44 35.52409492282702
45 35.39918483221905
46 35.27080789753761
47 35.13887524686

In [12]:
def sample(start_char):

    h=np.zeros((hidden_size,1))

    x=np.zeros((vocab_size,1))
    x[char_to_idx[start_char]]=1

    result=start_char

    for _ in range(20):

        h=np.tanh(
            np.dot(Wxh,x)
            +
            np.dot(Whh,h)
            +
            bh
        )

        y=np.dot(Why,h)+by

        p=softmax(y)

        idx=np.argmax(p)

        char=idx_to_char[idx]

        if char=="<END>":
            break

        result += char

        x=np.zeros((vocab_size,1))
        x[idx]=1

    return result

In [13]:
print(sample("a"))

anjali
